# V2 Task 2 factor ensemble

This is a new, independent Task 2 pipeline. Existing files and checkpoints are not modified. It uses only `train.xlsx` and creates no leaderboard submission.

V2 directly addresses the first submission failure:

- five user-grouped out-of-fold splits;
- word and character TF-IDF for explicit rare-factor phrases;
- one balanced classifier per factor;
- cross-fitted co-occurrence/risk meta-model;
- rank-based prevalence calibration with rare-label caps;
- no single-fold probability threshold such as `0.05`.

In [1]:
from pathlib import Path
import importlib
import json
import pandas as pd
import v2_factor_ensemble as v2
v2 = importlib.reload(v2)

TRAIN_PATH = Path('train.xlsx')
assert TRAIN_PATH.exists()

## Run five-fold out-of-fold evaluation

Run the next cell. This is CPU/sparse training and should be much faster than transformer fine-tuning. It does not load `leaderboard.xlsx`.

In [2]:
cfg = v2.V2Config(
    folds=5,
    seed=42,
    word_features=80_000,
    char_features=120_000,
    prevalence_cap_multiplier=2.5,
    quota_shrink_to_gold=0.35,
    output_dir='outputs/v2_factor_ensemble',
)
metrics = v2.run_v2_oof(TRAIN_PATH, cfg)
print('Raw 0.5 Macro F1:', metrics['raw_0.5_macro_f1'])
print('Fixed-prevalence Macro F1:', metrics['fixed_prevalence_macro_f1'])
print('Calibrated Macro F1:', metrics['calibrated_macro_f1'])
print('Gold labels/post:', metrics['average_gold_labels'])
print('Calibrated labels/post:', metrics['average_calibrated_labels'])

Fold 1: raw 0.5 Macro F1=0.3598
Fold 2: raw 0.5 Macro F1=0.3219
Fold 3: raw 0.5 Macro F1=0.3640
Fold 4: raw 0.5 Macro F1=0.3771
Fold 5: raw 0.5 Macro F1=0.3504
Raw 0.5 Macro F1: 0.3595784252685675
Fixed-prevalence Macro F1: 0.4322305547098659
Calibrated Macro F1: 0.4406277723021574
Gold labels/post: 2.9186544342507643
Calibrated labels/post: 3.2966360856269112


## Inspect every factor

The critical checks are `submission_rate`, rare-label precision, and Macro F1. A rare factor must not receive a submission rate remotely close to 100%.

In [3]:
per_label = pd.read_csv('outputs/v2_factor_ensemble/oof_per_label.csv')
display(per_label.sort_values('f1'))
display(per_label.sort_values('gold_rate').head(10)[
    ['factor', 'support', 'gold_rate', 'submission_rate', 'precision', 'recall', 'f1']
])

,factor,support,gold_rate,submission_rate,blend_meta_weight,precision,recall,f1
18,sexual orientation related issues,8,0.004893,0.004281,0.7,0.142857,0.125000,0.133333
16,cognitive deficits,33,0.020183,0.021407,0.3,0.142857,0.151515,0.147059
13,exposure to others' suicide,14,0.008563,0.007951,0.3,0.153846,0.142857,0.148148
2,substance use,33,0.020183,0.020183,0.0,0.212121,0.212121,0.212121
23,meaning in life,45,0.027523,0.033028,0.4,0.203704,0.244444,0.222222
6,poor school performance,16,0.009786,0.011621,0.2,0.210526,0.250000,0.228571
22,sense of responsibility,58,0.035474,0.028746,0.3,0.319149,0.258621,0.285714
1,physical health/characteristic,78,0.047706,0.093578,0.1,0.241830,0.474359,0.320346
7,low socio-economic status,54,0.033028,0.038532,0.0,0.349206,0.407407,0.376068
19,social support,112,0.068502,0.070336,0.4,0.373913,0.383929,0.378855


,factor,support,gold_rate,submission_rate,precision,recall,f1
18,sexual orientation related issues,8,0.004893,0.004281,0.142857,0.125000,0.133333
13,exposure to others' suicide,14,0.008563,0.007951,0.153846,0.142857,0.148148
6,poor school performance,16,0.009786,0.011621,0.210526,0.250000,0.228571
2,substance use,33,0.020183,0.020183,0.212121,0.212121,0.212121
16,cognitive deficits,33,0.020183,0.021407,0.142857,0.151515,0.147059
23,meaning in life,45,0.027523,0.033028,0.203704,0.244444,0.222222
7,low socio-economic status,54,0.033028,0.038532,0.349206,0.407407,0.376068
22,sense of responsibility,58,0.035474,0.028746,0.319149,0.258621,0.285714
15,traumatic experience,64,0.039144,0.035474,0.551724,0.500000,0.524590
1,physical health/characteristic,78,0.047706,0.093578,0.241830,0.474359,0.320346


## Decision gate

Save the notebook after the preceding cells finish and ask Codex to inspect it. No V2 leaderboard or combined submission code will be added until the out-of-fold results and prediction rates pass review.

Later combination will be explicit:

- `risk_level` and `evidence`: preserved Task 1 transformer;
- `factors`: V2 factor ensemble;
- final CSV filename: `SIT-MSF.csv`.